# 5. 提出ファイルを作る

04 が出した446人の生存確率を、投稿できる形式に整える。

やることは形式を合わせるだけだが、ここを間違えると
**中身が正しくても点数が出ない**。確認しながら進める。

In [ ]:
import pandas as pd

# 04 が保存した予測
pred = pd.read_csv("../data/processed/pred_test.csv", index_col=0)

# 提出フォーマットの見本。header=None は「1 行目は列名ではなくデータ」の指定
sample = pd.read_csv("../data/raw/sample_submit.csv", index_col=0, header=None)

print("pred:", pred.shape, " sample:", sample.shape)

## 1. 提出フォーマットを確かめる

見本ファイルの中身はこうなっている。

```
0,0
1,1
2,0
```

読み取れる決まりが3つある。

- **ヘッダ行が無い** — 1行目からデータが始まる
- **1列目が id** — 評価用データの id と一致していること
- **2列目が予測値**

見本の2列目は `0` と `1` だが、これはあくまで見本の値。
仕様上は**生存確率（0.0〜1.0 の小数）**を入れる。

In [ ]:
# 見本を読み込むと、列名が 1、index 名が 0 になる
#   ヘッダが無いので、pandas が位置の番号をそのまま名前にしている
print("列名:", list(sample.columns), " index 名:", sample.index.name)
print("見本に入っている値:", sorted(sample[1].unique()))
print()
print("id が pred と完全一致:", list(sample.index) == list(pred.index))

# 出力の見方
#   id の一致は必ず確認する。ここがずれていると、別人の予測を出すことになる

## 2. 評価指標は AUC

この課題の採点には **AUC** が使われる。名前は **A**rea **U**nder the **C**urve
（曲線の下の面積）の略だが、名前から中身は想像できない。
実際にやっているのは「**順位が正しく並んでいるか**」の測定。

### 4人で考える

生存者2人・死亡者2人を、モデルが付けた確率の高い順に並べる。

```
確率    実際
0.969   生存 ○
0.626   死亡 ×    ← この人が上に来てしまった
0.528   生存 ○
0.121   死亡 ×
```

理想は生存者が全員上に来ること。次に、生存者と死亡者を1人ずつ組にして総当たりする。

| 生存者 | 死亡者 | 生存者の方が上か |
| --- | --- | --- |
| 0.969 | 0.626 | ○ |
| 0.969 | 0.121 | ○ |
| 0.528 | 0.626 | **×** |
| 0.528 | 0.121 | ○ |

4組中3組が正しい順序なので、AUC は 3 / 4 = **0.75**。
つまり「**生存した人の方に高い確率を付けられた割合**」を表す。

| AUC | 意味 |
| --- | --- |
| 1.0 | 生存者が全員上。完璧 |
| 0.5 | でたらめ。全員に同じ値を出しても 0.5 になる |
| 0.5 未満 | 逆に並べた方がマシ。生存確率と死亡確率を取り違えていないか疑う |

正解率のときの基準は「全員が助からないと答えた場合の 0.5955」だったが、
AUC の基準は **0.5** になる。

### 正解は二択のまま。採点方法が二択を求めていない

ここで混乱しやすい点がある。正解は `0` か `1` の記録しかないのに、
なぜ小数を提出するのか。

採点する側は、提出された確率を 0/1 に**変換しない**。代わりに
「正解が `1` の人たちの方に、大きい数字が付いているか」を数える。
大小が付く数字であれば何でも採点できるので、0/1 である必要がない。

健康診断にたとえると分かりやすい。病気かどうかは二択だが、検査結果は血糖値のような
連続した数値で出てくる。「180 以上なら病気」と線を引かなくても、
**病気の人の方に高い値が出ているか**を見れば検査の性能は測れる。
AUC はもともと、こうした検査の性能評価で使われてきた指標。

### 順序さえ同じなら、値は何でもよい

AUC が順位しか見ていないことは、値を変えて試すと分かる。

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
import numpy as np

# 04 と同じ手順で検証用データを用意する
train = pd.read_csv("../data/processed/train_processed.csv", index_col=0)
X = train.drop(columns=["survived"]); y = train["survived"]
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)
m = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_train, y_train)
prob = m.predict_proba(X_valid)[:, 1]

print("順序を保ったまま値を変える")
print("  確率そのまま  :", round(roc_auc_score(y_valid, prob), 4))
print("  100 倍する    :", round(roc_auc_score(y_valid, prob * 100), 4))
print("  2 乗して歪める:", round(roc_auc_score(y_valid, prob ** 2), 4))
print()
print("順序を壊す")
print("  0/1 に丸める  :", round(roc_auc_score(y_valid, (prob > 0.5).astype(int)), 4))
print("  大小を逆にする:", round(roc_auc_score(y_valid, -prob), 4))

# 出力の見方
#   上の 3 つは全て同じ値。値の大きさや単位は一切見ていない
#   2 乗すると「0.7」が「0.49」になり値としては不正確になるが、
#   順序が変わらないので AUC は動かない
#   逆にすると 0.5 を大きく下回る。もし提出してこうなったら、
#   predict_proba の列を取り違えていないか確認する

### だから 0/1 に丸めない

`0` と `1` を並べたファイルも提出はできるが、点数が落ちる。
同じ数字どうしの比較が大量に発生し、大小が付かず引き分けになるため。

In [ ]:
hard = (prob > 0.5).astype(int)
pos, neg = prob[y_valid.values == 1], prob[y_valid.values == 0]
pos_h, neg_h = hard[y_valid.values == 1], hard[y_valid.values == 0]

print("検証用89人のペア総数:", len(pos) * len(neg), "組")
print("  確率のまま: 引き分け", sum(1 for a in pos for b in neg if a == b), "組")
print("  0/1 に丸めた後: 引き分け", sum(1 for a in pos_h for b in neg_h if a == b), "組")
print()
print("AUC  確率のまま:", round(roc_auc_score(y_valid, prob), 4),
      " 0/1:", round(roc_auc_score(y_valid, hard), 4))

# 出力の見方
#   丸めると 1908 組のうち 837 組（44%）が引き分けに変わる
#   正しく並べられていた順序が判定不能になるため、0.81 から 0.73 まで落ちる

### AUC が見ていないこと

AUC は順位しか見ないので、**確率の値そのものが正確かどうかは測っていない**。
上で 2 乗しても AUC が変わらなかったのは、そのため。

値の正確さ（「0.7 と言った人が本当に7割助かるか」）を知りたい場合は、
確率帯ごとの実際の生存率を並べて確かめる。

In [ ]:
d = pd.DataFrame({"p": prob, "y": y_valid.values})
d["帯"] = pd.cut(d["p"], [0, 0.2, 0.4, 0.6, 0.8, 1.0])
d.groupby("帯", observed=True).agg(
    人数=("y", "size"), 予測の平均=("p", "mean"), 実際の生存率=("y", "mean")
).round(3)

# 出力の見方
#   0.887 くらいと予測した 15 人のうち、実際に 93.3% が生存している。
#   値としてもおおむね合っている
#   ずれが目立つのは 0.4〜0.6（10 人）と 0.2〜0.4（14 人）の帯。
#   どちらも人数が少なく、数人の生死で大きく動くので誤差の範囲

## 3. 見本の値を置き換える

見本の2列目を、自分の予測で上書きする。
`sample` の id と `pred` の id が一致していることは1節で確認済み。

In [ ]:
# sample[1] が予測値の列。ここを pred の中身に差し替える
#   .values で数値の並びだけを取り出している。
#   DataFrame のまま代入すると、列名の違いで噛み合わないことがある
sample[1] = pred["pred"].values

sample.head()

# 出力の見方
#   2 列目が 0/1 から小数に変わっている
#   左端の id は見本のものをそのまま使っているので、順序も仕様どおり

## 4. ヘッダ無しで書き出す

`to_csv` は既定で列名を1行目に書いてしまう。仕様に合わせて抑制する。

In [ ]:
# header=False で列名の行を書かない
#   index=True で id を残す（既定なので省略してもよいが、意図として明示する）
sample.to_csv("../data/submissions/submit.csv", header=False, index=True)

print("書き出した")

In [ ]:
# 書き出したファイルをそのまま読み直して、仕様どおりか確認する
#   自分が持っている変数ではなくファイルを見るのが要点。
#   書き出す処理そのものが間違っていた場合、変数を見ても気づけない
check = pd.read_csv("../data/submissions/submit.csv", header=None)

print("行数:", len(check), "（446 行であること）")
print("列数:", check.shape[1], "（2 列であること）")
print("id が評価用データと一致:", check[0].tolist() == list(pred.index))
print("予測値の範囲:", round(check[1].min(), 4), "〜", round(check[1].max(), 4), "（0〜1 に収まること）")
print("欠損:", int(check.isna().sum().sum()), "（0 であること）")

# ヘッダの有無は、ファイルの 1 行目を文字として見るのが確実
with open("../data/submissions/submit.csv") as f:
    print("ファイルの 1 行目:", f.readline().strip(), "（列名ではなくデータであること）")

## 5. 投稿する

`data/submissions/submit.csv` をコンペの投稿ページからアップロードする。
しばらくすると AUC での評価結果が通知される。

手元の検証用データでの AUC は 0.81 前後だったので、その付近が目安になる。
大きく下回った場合は、提出形式のどこかが崩れている可能性を先に疑う。

## この章のまとめ

- 提出形式は**ヘッダ無し**・1列目が id・2列目が予測値
- 評価指標が AUC なので、**確率のまま出す**。0/1 に丸めると
  検証用データで 0.81 → 0.73 まで落ちた
- 見本ファイルの id をそのまま使い、2列目だけを差し替えた。
  id を自分で作り直すと、並び順の取り違えで別人の予測を出す危険がある
- 書き出した後は**ファイルを読み直して**行数・列数・1行目・id・値の範囲を確認した

ここまでで一通りの流れが終わった。精度を上げたい場合の次の一手は、
04 で見たとおり `age` の扱いを変える、欠損の埋め方を見直す、
別のモデルを試す、といった方向になる。